# Aligned Translation Surprisal Predicts Source-Text Eye Movements During Sight Translation

This notebook reproduces the mixed-effects model and cross-validation analyses reported in the paper.

**Data**: Eye-tracking data from the EMMT corpus (Bhattacharya et al., 2022). Set `DATA_DIR` below to the folder containing the preprocessed CSV files.

## Setup

In [ ]:
suppressPackageStartupMessages({
  library(lme4)
  library(lmerTest)
  library(dplyr)
  library(ggplot2)
})

DATA_DIR    <- "data"      # folder containing the preprocessed CSV files
FIGURES_DIR <- "figures"   # output folder for figures
dir.create(FIGURES_DIR, showWarnings=FALSE)

ctrl    <- lmerControl(optimizer="bobyqa")
c_scale <- function(x) (x - mean(x, na.rm=TRUE)) / sd(x, na.rm=TRUE)
se      <- function(x) sd(x) / sqrt(length(x))

## Data Loading

In [ ]:
fix  <- read.csv(file.path(DATA_DIR, "fixation_durations_word.csv"),   stringsAsFactors=FALSE)
soft <- read.csv(file.path(DATA_DIR, "nmt_surprisal_soft_word.csv"),   stringsAsFactors=FALSE)
mono <- read.csv(file.path(DATA_DIR, "monolingual_surprisal_word.csv"), stringsAsFactors=FALSE)
freq <- read.table(file.path(DATA_DIR, "subtlex_us.csv"), sep="\t", header=TRUE,
                   stringsAsFactors=FALSE, quote="") %>%
  select(word=Word, log10_freq=Lg10WF) %>%
  mutate(word=tolower(trimws(word)))

## Data Preparation

In [ ]:
sent_lengths <- soft %>%
  group_by(sentence_id) %>%
  summarise(sent_len=max(word_index)+1, .groups="drop")

predictors <- soft %>%
  left_join(sent_lengths, by="sentence_id") %>%
  mutate(word_length   = nchar(word),
         word_position = word_index / (sent_len - 1),
         word_lower    = tolower(trimws(word))) %>%
  left_join(freq, by=c("word_lower"="word")) %>%
  select(sentence_id, word_index, word_length, word_position,
         nmt_soft=surprisal_soft, log10_freq) %>%
  left_join(mono %>% select(sentence_id, word_index, mono_surprisal=surprisal_sum),
            by=c("sentence_id","word_index"))

df <- fix %>%
  left_join(predictors, by=c("sentence_id","word_index")) %>%
  mutate(log_tfd=log(total_fixation_duration_ms))

df_soft_all <- df %>%
  filter(stage=="translate",
         !is.na(nmt_soft), !is.na(mono_surprisal), !is.na(log10_freq)) %>%
  mutate(c_nmt  = c_scale(nmt_soft),
         c_mono = c_scale(mono_surprisal),
         c_wlen = c_scale(word_length),
         c_wpos = c_scale(word_position),
         c_freq = c_scale(log10_freq))

cat("n (translate stage, all words) =", nrow(df_soft_all), "\n")

## Primary Mixed-Effects Model

Attempts a random slope for `c_nmt` by participant; falls back to intercept-only if singular.

In [ ]:
m <- lmer(log_tfd ~ c_nmt + c_wlen + c_wpos + c_freq +
            (1 + c_nmt | participant) + (1 | sentence_id),
          data=df_soft_all, REML=FALSE, control=ctrl)

if (isSingular(m)) {
  cat("Random slope singular — dropping to intercept only\n")
  m <- lmer(log_tfd ~ c_nmt + c_wlen + c_wpos + c_freq +
              (1 | participant) + (1 | sentence_id),
            data=df_soft_all, REML=FALSE, control=ctrl)
}

cat("=== Fixed effects ===\n")
print(summary(m)$coefficients, digits=3)

## Cross-Validation

10-fold sentence-level CV following Lim et al. (2024). Each predictor is evaluated as the mean per-observation difference in held-out log-likelihood (Δllh) against a controls-only baseline. Significance is tested by a paired permutation test (1,000 permutations).

In [ ]:
all_sids <- sort(unique(df_soft_all$sentence_id))

held_out_llh <- function(model, test_data) {
  mu <- predict(model, newdata=test_data,
                re.form=~(1|participant), allow.new.levels=TRUE)
  mean(dnorm(test_data$log_tfd, mean=mu, sd=sigma(model), log=TRUE))
}

base_f <- log_tfd ~ c_wlen + c_wpos + c_freq + (1|participant) + (1|sentence_id)
nmt_f  <- update(base_f, . ~ . + c_nmt)
mono_f <- update(base_f, . ~ . + c_mono)

perm_p <- function(deltas, n_perm=1000, seed=42) {
  set.seed(seed); obs <- mean(deltas)
  mean(replicate(n_perm,
    mean(deltas * sample(c(-1,1), length(deltas), replace=TRUE))) >= obs)
}

run_folds <- function(fold_seed) {
  set.seed(fold_seed)
  fold_ids <- setNames(sample(rep(1:10, length.out=length(all_sids))), all_sids)
  suppressWarnings(sapply(1:10, function(k) {
    ts    <- names(fold_ids)[fold_ids == k]
    train <- df_soft_all %>% filter(!sentence_id %in% ts)
    test  <- df_soft_all %>% filter( sentence_id %in% ts)
    b <- held_out_llh(lmer(base_f, data=train, REML=FALSE, control=ctrl), test)
    c(nmt  = held_out_llh(lmer(nmt_f,  data=train, REML=FALSE, control=ctrl), test) - b,
      mono = held_out_llh(lmer(mono_f, data=train, REML=FALSE, control=ctrl), test) - b)
  }))
}

In [ ]:
cat("Running primary 10-fold CV (seed=99) ...\n")
folds_main  <- run_folds(99)
d_nmt_main  <- folds_main["nmt",]
d_mono_main <- folds_main["mono",]

cat("\n=== Primary CV result (seed=99) ===\n")
cat(sprintf("  c_nmt : Δllh=%.5f  SE=%.5f  p=%.3f\n",
            mean(d_nmt_main),  se(d_nmt_main),  perm_p(d_nmt_main)))
cat(sprintf("  c_mono: Δllh=%.5f  SE=%.5f  p=%.3f\n",
            mean(d_mono_main), se(d_mono_main), perm_p(d_mono_main)))

## Robustness Check: Repeated 10-Fold CV (50 Seeds)

During implementation, we found that permutation p-values varied considerably across fold assignments. With only 200 sentences and ten folds, a few discordant folds are enough to shift the result across the significance threshold. The procedure is therefore repeated 50 times with different random seeds.

In [ ]:
one_rep <- function(seed) {
  folds <- run_folds(seed)
  list(p_nmt  = perm_p(folds["nmt",]),
       p_mono = perm_p(folds["mono",]),
       d_nmt  = mean(folds["nmt",]),
       d_mono = mean(folds["mono",]))
}

cat("Running robustness check (50 seeds) ...\n")
t0 <- proc.time()["elapsed"]
results <- lapply(1:50, function(i) {
  if (i %% 10 == 0)
    cat(sprintf("  seed %d/50  (%.0fs elapsed)\n", i, proc.time()["elapsed"] - t0))
  one_rep(i)
})
cat(sprintf("  done (%.0fs)\n", proc.time()["elapsed"] - t0))

ps_nmt  <- sapply(results, `[[`, "p_nmt")
ps_mono <- sapply(results, `[[`, "p_mono")
ds_nmt  <- sapply(results, `[[`, "d_nmt")
ds_mono <- sapply(results, `[[`, "d_mono")

cat("\n=== Robustness check (50 seeds) ===\n")
cat(sprintf("  c_nmt : mean Δllh=%.5f  prop<.05=%d/50  median p=%.3f\n",
            mean(ds_nmt), sum(ps_nmt < .05), median(ps_nmt)))
cat(sprintf("  c_mono: mean Δllh=%.5f  prop<.05=%d/50  median p=%.3f\n",
            mean(ds_mono), sum(ps_mono < .05), median(ps_mono)))

## Figures

In [ ]:
# Figure: Δllh comparison (primary CV, seed=99)
pred_labels <- c("c[nmt]"  = expression(italic(c)[nmt]),
                 "c[mono]" = expression(italic(c)[mono]))

df_plot <- data.frame(
  predictor = factor(c("c[nmt]", "c[mono]"), levels=c("c[nmt]", "c[mono]")),
  delta     = c(mean(d_nmt_main),  mean(d_mono_main)),
  lo        = c(mean(d_nmt_main) - 1.96*se(d_nmt_main),
                mean(d_mono_main) - 1.96*se(d_mono_main)),
  hi        = c(mean(d_nmt_main) + 1.96*se(d_nmt_main),
                mean(d_mono_main) + 1.96*se(d_mono_main)),
  sig       = c(perm_p(d_nmt_main) < .05, perm_p(d_mono_main) < .05)
)

p_fig <- ggplot(df_plot, aes(x=predictor, y=delta, colour=sig)) +
  geom_point(size=3.5) +
  geom_errorbar(aes(ymin=lo, ymax=hi), width=0.08, linewidth=0.7) +
  geom_hline(yintercept=0, linetype="dashed", colour="#E8A000", linewidth=0.7) +
  scale_colour_manual(values=c("TRUE"="#2980B9","FALSE"="grey60"),
                      labels=c("TRUE"="p < .05","FALSE"="n.s."), name=NULL) +
  scale_x_discrete(labels=pred_labels) +
  labs(x=NULL, y=expression(Delta * "llh against controls")) +
  theme_bw(base_size=11) +
  theme(panel.grid.minor=element_blank(),
        panel.grid.major.x=element_blank(),
        legend.position="bottom")

ggsave(file.path(FIGURES_DIR, "delta_llh_comparison.pdf"), p_fig, width=3.5, height=3.2)
print(p_fig)

In [ ]:
# Figure: p-value distribution (robustness check)
df_phist <- data.frame(
  p    = c(ps_nmt, ps_mono),
  pred = rep(c("c[nmt]", "c[mono]"), each=50)
)
df_phist$pred <- factor(df_phist$pred, levels=c("c[nmt]", "c[mono]"))

p_hist <- ggplot(df_phist, aes(x=p)) +
  geom_histogram(breaks=seq(0, 1, 0.05), fill="steelblue",
                 colour="white", alpha=0.85) +
  geom_vline(xintercept=0.05, linetype="dashed",
             colour="#C0392B", linewidth=0.8) +
  facet_wrap(~pred, nrow=1, labeller=label_parsed) +
  labs(x="Permutation p-value (per seed)",
       y="Count (out of 50 seeds)") +
  theme_bw(base_size=11) +
  theme(panel.grid.minor=element_blank(),
        strip.background=element_rect(fill="grey92", colour="grey70"))

ggsave(file.path(FIGURES_DIR, "pvalue_dist.pdf"), p_hist, width=5, height=2.8)
print(p_hist)